# mjo_cross

- "Calculate space-time cross spectrum over multiple segments"
- [NCL Reference](https://www.ncl.ucar.edu/Document/Functions/Diagnostics/mjo_cross.shtml)

### Example NCL Script and Output

- mjo_cross.ncl
- mjo_output/mjo_cross_output.txt

### Input Data
- X: (time, lat, lon)
- Y: (time, lat, lon)
- segLength: length of segment
- segOverlap: overlap of the segment

### Returns:
Three-dimensional array (16, wavenumber, frequency)

The return variable will be a three-dimensional array (16,wavenumber,frequency) containing the 16 cross spectral quantities averaged over all segments.

         ( 0,:,:)  -  symmetric power spectrum of x
         ( 1,:,:)  -  asymmetric power spectrum of x
         ( 2,:,:)  -  symmetric power spectrum of y
         ( 3,:,:)  -  asymmetric power spectrum of y
         ( 4,:,:)  -  symmetric cospectrum
         ( 5,:,:)  -  asymmetric cospectrum
         ( 6,:,:)  -  symmetric quadrature spectrum
         ( 7,:,:)  -  asymmetric quadrature spectrum
         ( 8,:,:)  -  symmetric coherence-squared spectrum
         ( 9,:,:)  -  asymmetric coherence-squared spectrum
         (10,:,:)  -  symmetric phase spectrum
         (11,:,:)  -  asymmetric phase spectrum
         (12,:,:)  -  symmetric component-1 phase spectrum
         (13,:,:)  -  asymmetric component-1 phase spectrum
         (14,:,:)  -  symmetric component-2 phase spectrum
         (15,:,:)  -  asymmetric component-2 phase spectrum


In [55]:
import os
import xarray as xr
import numpy as np
from scipy import signal # csd: cross-spectral density/power
from scipy.signal import welch # power spectral density
import xrft # xarray FFT

In [3]:
segLen = 256        # 256 time steps (256 days in daily data)
segOverLap = 50     # 15 time steps of overlap

In [4]:
latS = -15
latN = 15

time_start = "1979-01-01"
time_end = "1981-12-31"

In [5]:
u850_data = xr.open_dataset(os.getcwd() + "/data/anomaly/QBOi.EXP1.AMIP.001.u850.day.anom.nc")
u850_data = u850_data.sel(time=slice(time_start, time_end), lat=slice(latS, latN))
u850_data

<xarray.Dataset> Size: 40MB
Dimensions:  (time: 1095, lat: 32, lon: 288)
Coordinates:
  * time     (time) object 9kB 1979-01-01 00:00:00 ... 1981-12-31 00:00:00
  * lat      (lat) float64 256B -14.61 -13.66 -12.72 ... 12.72 13.66 14.61
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.3 357.5 358.8
Data variables:
    date     (time) float64 9kB ...
    U850     (time, lat, lon) float32 40MB ...

In [6]:
flut_data = xr.open_dataset(os.getcwd() + "/data/anomaly/QBOi.EXP1.AMIP.001.flut.day.anom.nc")
flut_data = flut_data.sel(time=slice(time_start, time_end), lat=slice(latS, latN))
flut_data

<xarray.Dataset> Size: 40MB
Dimensions:  (time: 1095, lat: 32, lon: 288)
Coordinates:
  * time     (time) object 9kB 1979-01-01 00:00:00 ... 1981-12-31 00:00:00
  * lat      (lat) float64 256B -14.61 -13.66 -12.72 ... 12.72 13.66 14.61
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.3 357.5 358.8
Data variables:
    date     (time) float64 9kB ...
    FLUT     (time, lat, lon) float32 40MB ...
Attributes:
    history:  Thu Mar  6 13:31:31 2025: ncatted -a cell_methods,FLUT,m,c,time...
    NCO:      netCDF Operators version 5.3.1 (Homepage = http://nco.sf.net, C...

#### Symmetric Power Spectrum of x and y

In [36]:
def welch_xarray_wrapper(v, **kwargs):
    sample_freqs, ps = welch(v, scaling="spectrum", **kwargs)
    return sample_freqs, ps

def sampling_frequency(dataset):
    # Determine sampling frequency
    time_deltas = u850_data.time.diff(dim="time")
    median_delta = time_deltas.median().values # account for NA values
    seconds_per_delta = median_delta.astype('timedelta64[s]').item().total_seconds()
    sample_freq_hz = 1 / seconds_per_delta
    return sample_freq_hz

In [48]:
# Compute the power spectrum along the time dimension

# X: FFT
#u850_xr = u850_data.to_dataarray(name="u850")
#u850_xr = u850_xr.chunk({'time': -1, 'lat': 100, 'lon': 100, 'variable': 100}) # chunking large dataset to run xrft
#fft_x = xrft.fft(u850_xr, dim=['time'])
#power_spectrum = np.abs(fft_x)**2
#power_spectrum

# X: Welch - Power Spectrum
u850_sf, u850_ps = xr.apply_ufunc(
    welch_xarray_wrapper,
    u850_data['U850'], # Apply to a specific DataArray within the Dataset
    input_core_dims=[['time']],
    output_core_dims=[["sample_freqs"], ["power_spectrum"]],
    vectorize=True,
    kwargs={'fs': sampling_frequency(u850_data), "nperseg": segLen, "noverlap": segOverLap}
)

In [49]:
u850_sf

<xarray.DataArray 'U850' (lat: 32, lon: 288, sample_freqs: 129)> Size: 10MB
array([[[0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06],
        [0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06],
        [0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06],
        ...,
        [0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06],
        [0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06],
        [0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06]],

       [[0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06],
        [0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06],
        [0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06],
...
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06],
        [0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06],
        [0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06]],

       [[0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06],
        [0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06],
        [0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06],
        ...,
        [0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06],
        [0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06],
        [0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06]]],
      shape=(32, 288, 129))
Coordinates:
  * lat      (lat) float64 256B -14.61 -13.66 -12.72 ... 12.72 13.66 14.61
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.3 357.5 358.8
Dimensions without coordinates: sample_freqs
Attributes:
    Sampling_Sequence:  rad_lwsw
    units:              m/s
    long_name:          Zonal wind at 850 mbar pressure surface
    cell_methods:       time: mean                                           ...

In [50]:
u850_ps

<xarray.DataArray 'U850' (lat: 32, lon: 288, power_spectrum: 129)> Size: 5MB
array([[[1.9768395e-01, 4.8737529e-01, 3.0388579e-01, ...,
         2.7885239e-03, 6.1283028e-03, 6.1131688e-03],
        [2.2641805e-01, 5.8128119e-01, 2.9727653e-01, ...,
         2.8070887e-03, 4.8684292e-03, 3.7886370e-03],
        [2.4773528e-01, 7.0999581e-01, 2.8287202e-01, ...,
         3.9853929e-03, 4.4274079e-03, 2.8758338e-03],
        ...,
        [1.1319914e-01, 3.0439416e-01, 2.0264530e-01, ...,
         7.4869008e-03, 6.3027409e-03, 4.6247225e-03],
        [1.3674219e-01, 3.4713989e-01, 2.3699667e-01, ...,
         6.1907344e-03, 6.8760398e-03, 6.6960985e-03],
        [1.6480030e-01, 4.0802163e-01, 2.7415836e-01, ...,
         4.1379491e-03, 6.8688057e-03, 7.4704704e-03]],

       [[2.4987714e-01, 6.3390887e-01, 2.3436432e-01, ...,
         1.3157109e-03, 5.6095729e-03, 6.5673520e-03],
        [2.6621753e-01, 7.3722577e-01, 2.2943315e-01, ...,
         1.6126679e-03, 5.2584736e-03, 5.3653354e-03],
        [2.6951563e-01, 8.5859692e-01, 2.4607344e-01, ...,
         2.6683230e-03, 5.0991327e-03, 5.3170631e-03],
...
         1.1430074e-02, 8.0377245e-03, 2.1251009e-03],
        [1.7967232e-02, 1.8562214e-01, 4.2906493e-01, ...,
         1.2090685e-02, 5.5276509e-03, 1.6576136e-03],
        [1.9813892e-02, 1.9011633e-01, 4.0515414e-01, ...,
         1.0834397e-02, 3.9689285e-03, 1.5540519e-03]],

       [[2.1098968e-02, 2.5192049e-01, 5.0528878e-01, ...,
         6.5377154e-03, 3.8973598e-03, 1.8585310e-03],
        [1.8868241e-02, 2.5963193e-01, 5.2393943e-01, ...,
         5.4090647e-03, 3.9962139e-03, 1.6609363e-03],
        [1.8859401e-02, 2.7230582e-01, 5.2310175e-01, ...,
         3.3416396e-03, 2.7672593e-03, 9.6450298e-04],
        ...,
        [2.2097709e-02, 2.1722741e-01, 5.4399490e-01, ...,
         1.1331102e-02, 5.2376953e-03, 8.5023639e-04],
        [2.4286216e-02, 2.3636657e-01, 5.0097400e-01, ...,
         1.0960632e-02, 3.6363690e-03, 4.7308154e-04],
        [2.3732184e-02, 2.4401592e-01, 4.8743898e-01, ...,
         8.0875950e-03, 2.4646572e-03, 7.9642556e-04]]],
      shape=(32, 288, 129), dtype=float32)
Coordinates:
  * lat      (lat) float64 256B -14.61 -13.66 -12.72 ... 12.72 13.66 14.61
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.3 357.5 358.8
Dimensions without coordinates: power_spectrum
Attributes:
    Sampling_Sequence:  rad_lwsw
    units:              m/s
    long_name:          Zonal wind at 850 mbar pressure surface
    cell_methods:       time: mean                                           ...

In [51]:
# Compute the power spectrum along the time dimension

# Y:
#flut_xr = flut_data.to_dataarray(name="flut")
#flut_xr = flut_xr.chunk({'time': -1, 'lat': 100, 'lon': 100, 'variable': 100}) # chunking large dataset to run xrft
#fft_x = xrft.fft(flut_xr, dim=['time'])
#power_spectrum = np.abs(fft_x)**2
#power_spectrum

# X: Welch - Power Spectrum
flut_sf, flut_ps = xr.apply_ufunc(
    welch_xarray_wrapper,
    flut_data['FLUT'], # Apply to a specific DataArray within the Dataset
    input_core_dims=[['time']],
    output_core_dims=[["sample_freqs"], ["power_spectrum"]],
    vectorize=True,
    kwargs={'fs': sampling_frequency(flut_data), "nperseg": segLen, "noverlap": segOverLap}
)

In [53]:
flut_sf

<xarray.DataArray 'FLUT' (lat: 32, lon: 288, sample_freqs: 129)> Size: 10MB
array([[[0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06],
        [0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06],
        [0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06],
        ...,
        [0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06],
        [0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06],
        [0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06]],

       [[0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06],
        [0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06],
        [0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06],
...
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06],
        [0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06],
        [0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06]],

       [[0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06],
        [0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06],
        [0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06],
        ...,
        [0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06],
        [0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06],
        [0.00000000e+00, 4.52112269e-08, 9.04224537e-08, ...,
         5.69661458e-06, 5.74182581e-06, 5.78703704e-06]]],
      shape=(32, 288, 129))
Coordinates:
  * lat      (lat) float64 256B -14.61 -13.66 -12.72 ... 12.72 13.66 14.61
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.3 357.5 358.8
Dimensions without coordinates: sample_freqs
Attributes:
    cell_methods:       time: mean
    long_name:          Upwelling longwave flux at top of model
    units:              W/m2
    Sampling_Sequence:  rad_lwsw

In [54]:
flut_ps

<xarray.DataArray 'FLUT' (lat: 32, lon: 288, power_spectrum: 129)> Size: 5MB
array([[[1.16615760e+00, 8.04400826e+00, 5.61355257e+00, ...,
         4.75574620e-02, 7.46610686e-02, 3.47498506e-02],
        [1.24210179e+00, 8.20050335e+00, 5.17941189e+00, ...,
         5.20572662e-02, 9.12133604e-02, 3.53845395e-02],
        [1.74293816e+00, 9.28240967e+00, 4.58127642e+00, ...,
         6.95299581e-02, 1.20308019e-01, 4.04863469e-02],
        ...,
        [8.46752405e-01, 6.35256481e+00, 3.70664048e+00, ...,
         4.96856496e-02, 4.96632531e-02, 2.74511650e-02],
        [1.05898094e+00, 7.46129465e+00, 4.84303427e+00, ...,
         5.32401614e-02, 4.04190049e-02, 3.10468394e-02],
        [1.10683715e+00, 7.80311251e+00, 5.10436296e+00, ...,
         7.81836659e-02, 9.81003493e-02, 5.46517372e-02]],

       [[9.72117782e-01, 8.24426365e+00, 5.05683470e+00, ...,
         8.51820186e-02, 1.39368877e-01, 5.18944040e-02],
        [1.15400183e+00, 8.89294052e+00, 5.09466314e+00, ...,
         1.06533051e-01, 1.57502681e-01, 5.53465411e-02],
        [1.85491276e+00, 1.07243071e+01, 4.88440752e+00, ...,
         8.52062926e-02, 1.35602608e-01, 5.50229512e-02],
...
         2.32840014e+00, 1.32142186e+00, 7.21335828e-01],
        [7.49230499e+01, 2.57976685e+02, 6.48254318e+01, ...,
         1.92280853e+00, 1.04709077e+00, 7.82010615e-01],
        [6.49399643e+01, 2.43953781e+02, 5.68878860e+01, ...,
         1.32996690e+00, 9.38022733e-01, 7.00654864e-01]],

       [[4.85097771e+01, 2.18123642e+02, 6.38311195e+01, ...,
         1.62247443e+00, 1.02780676e+00, 5.35407007e-01],
        [3.95365715e+01, 2.09563019e+02, 6.28168564e+01, ...,
         1.32187188e+00, 7.47241855e-01, 3.51336807e-01],
        [3.31015892e+01, 1.90457870e+02, 6.18381195e+01, ...,
         1.78846443e+00, 1.58999407e+00, 3.64743143e-01],
        ...,
        [8.47974625e+01, 2.64659271e+02, 9.22154999e+01, ...,
         3.35084319e+00, 2.10418272e+00, 1.15348268e+00],
        [6.95236282e+01, 2.44951080e+02, 8.35990524e+01, ...,
         2.35802150e+00, 1.70723939e+00, 1.12454653e+00],
        [5.65948563e+01, 2.23587708e+02, 7.49469986e+01, ...,
         1.78016496e+00, 1.74276912e+00, 7.11476266e-01]]],
      shape=(32, 288, 129), dtype=float32)
Coordinates:
  * lat      (lat) float64 256B -14.61 -13.66 -12.72 ... 12.72 13.66 14.61
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.3 357.5 358.8
Dimensions without coordinates: power_spectrum
Attributes:
    cell_methods:       time: mean
    long_name:          Upwelling longwave flux at top of model
    units:              W/m2
    Sampling_Sequence:  rad_lwsw

## Previous:

In [31]:
freqs, cps = signal.csd(u850_data["U850"], flut_data["FLUT"],
                        nperseg=segLen, noverlap=segOverLap,
                       scaling="spectrum") # cross power spectrum

In [32]:
cps_mag = np.abs(cps)
cps_phase = np.angle(cps, deg=True)
cps_mag

array([[[3.59252751e-01, 5.51712751e+00, 1.56607037e+01, ...,
         2.07582125e-04, 2.95636331e-04, 3.63290630e-04],
        [1.78589419e-01, 8.70939064e+00, 1.77018108e+01, ...,
         3.67933942e-04, 4.74397180e-04, 1.64826357e-04],
        [7.88657665e-01, 1.44027643e+01, 2.07164688e+01, ...,
         1.74545639e-04, 1.42560180e-04, 4.50356602e-05],
        ...,
        [5.03039122e-01, 6.67679024e+00, 1.13200035e+01, ...,
         2.84013047e-04, 4.52353997e-04, 8.89952207e-05],
        [4.30214822e-01, 6.53344822e+00, 7.40704870e+00, ...,
         5.24572271e-04, 3.79076751e-04, 3.98082102e-06],
        [3.29580873e-01, 6.57409716e+00, 4.62748384e+00, ...,
         3.22538981e-04, 3.16020043e-04, 2.04730441e-05]],

       [[6.95979929e+00, 1.87492085e+01, 1.09296112e+01, ...,
         3.28140653e-04, 4.15842893e-04, 2.31772465e-05],
        [6.95932579e+00, 1.91676559e+01, 1.05645409e+01, ...,
         1.25167091e-04, 3.56738965e-05, 1.22104962e-06],
        [6.25880241e+00, 

In [6]:
print(u850_data["U850"].shape)
print(flut_data["FLUT"].shape)
print(f"time = {cps_mag.shape[0]}, lat = {cps_mag.shape[1]}, lon = {cps_mag.shape[2]}")

(2555, 192, 288)
(2555, 192, 288)
time = 2555, lat = 192, lon = 129


In [7]:
print(cps_mag.shape)
print(cps_phase.shape)
print(freqs.shape)

(2555, 192, 129)
(2555, 192, 129)
(129,)
